In [1]:
import torch
from transformers import AutoTokenizer, GPT2LMHeadModel, GPT2Config
from datasets import load_dataset

In [2]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token

config = GPT2Config(vocab_size=len(tokenizer))
model = GPT2LMHeadModel(config)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [3]:
DATASET_NAME = "elmurod1202/uzbek-sentiment-analysis"
ds = load_dataset(DATASET_NAME)

Repo card metadata block was not found. Setting CardData to empty.


In [4]:
def tokenize(examples):
    outputs = [text + tokenizer.eos_token for text in examples["text"]]
    return tokenizer(outputs)

train_ds = ds["train"].map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

In [5]:
block_size = 1024

def group_texts(examples):
    # Concatenate all the token arrays in the current batch
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    
    # Drop the small remainder of tokens at the very end of the dataset
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
        
    # Split the massive concatenated array into blocks of 'block_size'
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    
    # For training from scratch (causal modeling), the target labels are the exact same as the inputs.
    # The model handles shifting the tokens by one position internally during the forward pass.
    result["labels"] = result["input_ids"].copy()
    return result

train_ds = train_ds.map(
    group_texts,
    batched=True,
    batch_size=1024,
    num_proc=4
)

print(train_ds[0].keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [6]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=False
)

training_args = TrainingArguments(
    output_dir="./gpt2-trained-from-scratch",
    optim="adamw_torch_fused",
    learning_rate=2e-4,
    weight_decay=0.1,
    lr_scheduler_type='cosine',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    bf16=True,
    tf32=True,
    # fp16=True,
    num_train_epochs=10,
    # gradient_checkpointing=True,
    save_strategy="steps",
    logging_steps=100,
    save_steps=500
)

trainer = Trainer(
    model=model, 
    args=training_args,
    train_dataset=train_ds,
    data_collator=data_collator,
)

trainer.train()

/home/mardon/jupyterlab/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(
/home/mardon/jupyterlab/.venv/lib/python3.12/site-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)
`loss_type=None` was set in the config but it is unrecognized. Using the

Step,Training Loss
100,5.047500
200,3.419000
300,3.235000
400,3.140900
500,3.039200
600,2.947400
700,2.837300
800,2.719000
900,2.589400
1000,2.467600


TrainOutput(global_step=1280, training_loss=2.972100389003754, metrics={'train_runtime': 771.7563, 'train_samples_per_second': 13.217, 'train_steps_per_second': 1.659, 'total_flos': 5330357452800000.0, 'train_loss': 2.972100389003754, 'epoch': 10.0})

In [7]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
trainer.push_to_hub(f"gpt2-{DATASET_NAME.split('/')[-1]}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [9]:
trainer.save_model(f"./model/gpt2-{DATASET_NAME.split('/')[-1]}")
tokenizer.save_pretrained(f"./model/gpt2-{DATASET_NAME.split('/')[-1]}")

('./model/gpt2-uzbek-sentiment-analysis/tokenizer_config.json',
 './model/gpt2-uzbek-sentiment-analysis/special_tokens_map.json',
 './model/gpt2-uzbek-sentiment-analysis/vocab.json',
 './model/gpt2-uzbek-sentiment-analysis/merges.txt',
 './model/gpt2-uzbek-sentiment-analysis/added_tokens.json',
 './model/gpt2-uzbek-sentiment-analysis/tokenizer.json')

In [8]:
def generate_text(query, model, tokenizer, max_new_tokens=50, device=None):
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model.eval()
    model.to(device)
    
    inputs = tokenizer(query, return_tensors="pt").to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,      # How many new tokens to generate
            do_sample=True,                     # Enable sampling (adds randomness/creativity)
            temperature=0.8,                    # Higher = more random; Lower = more predictable
            top_k=50,                           # Limits sampling to the top 50 most likely tokens
            pad_token_id=tokenizer.eos_token_id # Prevents warnings since GPT-2 lacks a native pad token
        )
        
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    return generated_text

generate_text("salom", model, tokenizer, max_new_tokens=100)

"salomgan va men uni yoqtiraman, lekin menga to'g'ldim, buni u erda ishga qiloladi. Men o'ynashni xohlayman, shuning uchun bu so'ng yangi yoki iltimos, yoshqasamchilgan o'zgartira olmaydi. Uni yopilangi tuzatishni yoki planshetlaganimda"